PySpark Day 2 — DataFrame Transformations

Goal: practice narrow/wide transformations and build business-ready retail outputs.

Create/use a banking transactions DataFrame with: `transaction_id, customer_id, account_type, transaction_type, amount, city, transaction_date`.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.appName("BankingPysparkreport").getOrCreate()

banking_data = [
    (2001, 101, "Savings",  "Deposit",    55000.0, "Bengaluru", "2026-09-01"),
    (2002, 102, "Current",  "Withdrawal", 22000.0, "Hyderabad", "2026-09-01"),
    (2003, 103, "Savings",  "Transfer",    8000.0, "Chennai",   "2026-09-02"),
    (2004, 104, "Salary",   "Deposit",    55000.0, "Pune",      "2026-09-02"),
    (2005, 101, "Savings",  "Withdrawal",  5000.0, "Bengaluru", "2026-09-03"),
    (2006, 105, "Current",  "Deposit",    85000.0, "Mumbai",    "2026-09-03"),
    (2007, 106, "Savings",  "Transfer",   25000.0, "Delhi",     "2026-09-04"),
    (2008, 107, "Salary",   "Deposit",    60000.0, "Bengaluru", "2026-09-04"),
    (2009, 102, "Current",  "Transfer",   20000.0, "Hyderabad", "2026-09-05"),
    (2010, 108, "Savings",  "Withdrawal",  3000.0, "Chennai",   "2026-09-05"),
    (2011, 109, "Current",  "Deposit",    75000.0, "Pune",      "2026-09-06"),
    (2012, 110, "Savings",  "Deposit",    10000.0, "Mumbai",    "2026-09-06"),
    (2013, 103, "Savings",  "Withdrawal",  7000.0, "Chennai",   "2026-09-07"),
    (2014, 104, "Salary",   "Transfer",   18000.0, "Pune",      "2026-09-07"),
    (2015, 111, "Current",  "Withdrawal", 25000.0, "Delhi",     "2026-09-08"),
    (2016, 105, "Current",  "Transfer",   30000.0, "Mumbai",    "2026-09-08"),
]

cols=["transaction_id", "customer_id", "account_type", "transaction_type", "amount","city", "transaction_date"]

df=spark.createDataFrame(banking_data,cols)
df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|
|          2007|        106|     Savings|        Transfer|25000.0|    Delhi|      2026-09-04|
|          2008|        107|      Salary|         Deposit|60

1. Select transaction ID, customer ID and amount.

In [0]:

df.select("transaction_id","customer_id","amount").show()

+--------------+-----------+-------+
|transaction_id|customer_id| amount|
+--------------+-----------+-------+
|          2001|        101|55000.0|
|          2002|        102|22000.0|
|          2003|        103| 8000.0|
|          2004|        104|55000.0|
|          2005|        101| 5000.0|
|          2006|        105|85000.0|
|          2007|        106|25000.0|
|          2008|        107|60000.0|
|          2009|        102|20000.0|
|          2010|        108| 3000.0|
|          2011|        109|75000.0|
|          2012|        110|10000.0|
|          2013|        103| 7000.0|
|          2014|        104|18000.0|
|          2015|        111|25000.0|
|          2016|        105|30000.0|
+--------------+-----------+-------+



2. Filter transactions above ₹50,000.


In [0]:
df.filter(F.col("amount")> 50000).show()





+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|
|          2008|        107|      Salary|         Deposit|60000.0|Bengaluru|      2026-09-04|
|          2011|        109|     Current|         Deposit|75000.0|     Pune|      2026-09-06|
+--------------+-----------+------------+----------------+-------+---------+----------------+



3. Filter only `DEBIT` transactions.

In [0]:
df.filter(
    (F.col("transaction_type") == "Withdrawal") |
    (F.col("transaction_type") == "Transfer")
).show()


+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|
|          2007|        106|     Savings|        Transfer|25000.0|    Delhi|      2026-09-04|
|          2009|        102|     Current|        Transfer|20000.0|Hyderabad|      2026-09-05|
|          2010|        108|     Savings|      Withdrawal| 3000.0|  Chennai|      2026-09-05|
|          2013|        103|     Savings|      Withdrawal| 7000.0|  Chennai|      2026-09-07|
|          2014|        104|      Salary|        Transfer|18


4. Create `transaction_fee` as 1% of amount.


In [0]:
transaction_fee_df = df.withColumn("transaction_fee",F.col("amount")*0.01)
transaction_fee_df.show()



+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|transaction_fee|
+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|          550.0|
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|          220.0|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|           80.0|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|          550.0|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|           50.0|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|          850.0|
|         

In [0]:
transaction_fee_df.select("transaction_fee").show()

+---------------+
|transaction_fee|
+---------------+
|          550.0|
|          220.0|
|           80.0|
|          550.0|
|           50.0|
|          850.0|
|          250.0|
|          600.0|
|          200.0|
|           30.0|
|          750.0|
|          100.0|
|           70.0|
|          180.0|
|          250.0|
|          300.0|
+---------------+



5. Create `net_amount` after fee.


In [0]:
net_amount_df = transaction_fee_df.withColumn("net_amount_df",F.col("amount")-F.col("transaction_fee"))
net_amount_df.show()


+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+-------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|transaction_fee|net_amount_df|
+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+-------------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|          550.0|      54450.0|
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|          220.0|      21780.0|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|           80.0|       7920.0|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|          550.0|      54450.0|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|           50.0|       4950.0|
|       


6. Rename `city` to `branch_city`.


In [0]:
df.withColumnRenamed("city","branch_city").show()

+--------------+-----------+------------+----------------+-------+-----------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|branch_city|transaction_date|
+--------------+-----------+------------+----------------+-------+-----------+----------------+
|          2001|        101|     Savings|         Deposit|55000.0|  Bengaluru|      2026-09-01|
|          2002|        102|     Current|      Withdrawal|22000.0|  Hyderabad|      2026-09-01|
|          2003|        103|     Savings|        Transfer| 8000.0|    Chennai|      2026-09-02|
|          2004|        104|      Salary|         Deposit|55000.0|       Pune|      2026-09-02|
|          2005|        101|     Savings|      Withdrawal| 5000.0|  Bengaluru|      2026-09-03|
|          2006|        105|     Current|         Deposit|85000.0|     Mumbai|      2026-09-03|
|          2007|        106|     Savings|        Transfer|25000.0|      Delhi|      2026-09-04|
|          2008|        107|      Salary

7. Drop an unnecessary column.


In [0]:
df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|
|          2007|        106|     Savings|        Transfer|25000.0|    Delhi|      2026-09-04|
|          2008|        107|      Salary|         Deposit|60

In [0]:
df.drop("city").show()


+--------------+-----------+------------+----------------+-------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|transaction_date|
+--------------+-----------+------------+----------------+-------+----------------+
|          2001|        101|     Savings|         Deposit|55000.0|      2026-09-01|
|          2002|        102|     Current|      Withdrawal|22000.0|      2026-09-01|
|          2003|        103|     Savings|        Transfer| 8000.0|      2026-09-02|
|          2004|        104|      Salary|         Deposit|55000.0|      2026-09-02|
|          2005|        101|     Savings|      Withdrawal| 5000.0|      2026-09-03|
|          2006|        105|     Current|         Deposit|85000.0|      2026-09-03|
|          2007|        106|     Savings|        Transfer|25000.0|      2026-09-04|
|          2008|        107|      Salary|         Deposit|60000.0|      2026-09-04|
|          2009|        102|     Current|        Transfer|20000.0|      2026

8. Find distinct account types.

In [0]:
df.select("account_type").distinct().show()

+------------+
|account_type|
+------------+
|     Savings|
|     Current|
|      Salary|
+------------+



9. Remove duplicate transaction IDs.


In [0]:
df.dropDuplicates(["transaction_id"]).show()


+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|
|          2007|        106|     Savings|        Transfer|25000.0|    Delhi|      2026-09-04|
|          2008|        107|      Salary|         Deposit|60

10. Sort transactions by amount descending.


In [0]:
df.orderBy("amount", ascending=False).show()


+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|
|          2011|        109|     Current|         Deposit|75000.0|     Pune|      2026-09-06|
|          2008|        107|      Salary|         Deposit|60000.0|Bengaluru|      2026-09-04|
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|
|          2016|        105|     Current|        Transfer|30000.0|   Mumbai|      2026-09-08|
|          2007|        106|     Savings|        Transfer|25000.0|    Delhi|      2026-09-04|
|          2015|        111|     Current|      Withdrawal|25

11. Group by account type and calculate count, total amount, average amount, min and max.

In [0]:
df.groupBy("account_type").agg(F.count("*").alias("transaction_count"),
                               F.sum("amount").alias("total_amount"),
                               F.avg("amount").alias("average_amount"),
                               F.min("amount").alias("min_amount"),
                               F.max("amount").alias("max_amount")).show()

+------------+-----------------+------------+------------------+----------+----------+
|account_type|transaction_count|total_amount|    average_amount|min_amount|max_amount|
+------------+-----------------+------------+------------------+----------+----------+
|     Savings|                7|    113000.0|16142.857142857143|    3000.0|   55000.0|
|     Current|                6|    257000.0|42833.333333333336|   20000.0|   85000.0|
|      Salary|                3|    133000.0|44333.333333333336|   18000.0|   60000.0|
+------------+-----------------+------------+------------------+----------+----------+




12. Create `risk_level`: HIGH >= ₹100,000, MEDIUM >= ₹50,000, otherwise LOW.


In [0]:
from pyspark.sql import functions as F
risk_df= df.withColumn("risk_level",
                       F.when(F.col("amount") >= 100000, "HIGH")
                       .when(F.col("amount") >= 50000, "MEDIUM ")
                       .otherwise("LOW"))
risk_df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+----------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|risk_level|
+--------------+-----------+------------+----------------+-------+---------+----------------+----------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|   MEDIUM |
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|       LOW|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|       LOW|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|   MEDIUM |
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|       LOW|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|   MEDIUM |
|          2007|        106|     Savings|        Transf

13. Convert transaction date to DateType and extract year/month/day.


In [0]:


date_df = df.withColumn(
    "transaction_date",
    F.to_date(F.col("transaction_date"), "yyyy-MM-dd")
).withColumn(
    "year", F.year("transaction_date")
).withColumn(
    "month", F.month("transaction_date")
).withColumn(
    "day", F.dayofmonth("transaction_date")
)

date_df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+----+-----+---+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|year|month|day|
+--------------+-----------+------------+----------------+-------+---------+----------------+----+-----+---+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|2026|    9|  1|
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|2026|    9|  1|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|2026|    9|  2|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|2026|    9|  2|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|2026|    9|  3|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|2026|    9|  3|
|          2007|   

14. Handle null amount/city values appropriately.


In [0]:
handled_df = df.fillna({
    "amount": 0,
    "city": "Unknown"
})

handled_df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|
|          2007|        106|     Savings|        Transfer|25000.0|    Delhi|      2026-09-04|
|          2008|        107|      Salary|         Deposit|60

15. Create another DataFrame and perform `unionByName`.

In [0]:
data2 = [
    (2017, 112, "Savings", "Deposit", 45000.0, "Hyderabad", "2026-09-09"),
    (2018, 113, "Current", "Transfer", 30000.0, "Mumbai", "2026-09-09")
]

df2 = spark.createDataFrame(data2, df.columns)

df_union = df.unionByName(df2)

df_union.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|
|          2004|        104|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|
|          2007|        106|     Savings|        Transfer|25000.0|    Delhi|      2026-09-04|
|          2008|        107|      Salary|         Deposit|60

16. Create a customer master DataFrame and join it with transactions.


In [0]:
customer_data = [
    (101,"Praveen",  "Gold",     "Verified"),
    (102,"Meera Nair",  "Platinum", "Verified"),
    (103,"Ravi Shah",   "Silver",   "Verified"),
    (104,"Sneha Rao",   "Silver",   "Pending"),
    (105,"Imran Khan",  "Gold",     "Verified"),
    (106,"Divya Iyer",  "Silver",   "Verified"),
    (107,"Karthik S",   "Silver",   "Pending"),
    (108,"Anita Desai", "Platinum", "Verified"),
    (109,"Vijay Menon", "Gold",     "Verified"),
    (110,"Pooja Reddy", "Silver",   "Verified"),
    (111,"Rohit Jain",  "Gold",     "Verified"),
    (112,"Nisha Verma", "Platinum", "Verified"),
    (113,"Sameer Ali",  "Silver",   "Pending"),
]
customer_df = spark.createDataFrame(customer_data, ["customer_id","customer_name","customer_tier","kyc_status"])
df.join(customer_df, "customer_id", "left").orderBy("transaction_id").show()

+-----------+--------------+------------+----------------+-------+---------+----------------+-------------+-------------+----------+
|customer_id|transaction_id|account_type|transaction_type| amount|     city|transaction_date|customer_name|customer_tier|kyc_status|
+-----------+--------------+------------+----------------+-------+---------+----------------+-------------+-------------+----------+
|        101|          2001|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|      Praveen|         Gold|  Verified|
|        102|          2002|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|   Meera Nair|     Platinum|  Verified|
|        103|          2003|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|    Ravi Shah|       Silver|  Verified|
|        104|          2004|      Salary|         Deposit|55000.0|     Pune|      2026-09-02|    Sneha Rao|       Silver|   Pending|
|        101|          2005|     Savings|      Withdrawal| 5000.0|Ben

17. Repartition by account type and explain why.


In [0]:
repartitioned_df = df.repartition(3, "account_type")

repartitioned_df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          2001|        101|     Savings|         Deposit|55000.0|Bengaluru|      2026-09-01|
|          2002|        102|     Current|      Withdrawal|22000.0|Hyderabad|      2026-09-01|
|          2003|        103|     Savings|        Transfer| 8000.0|  Chennai|      2026-09-02|
|          2005|        101|     Savings|      Withdrawal| 5000.0|Bengaluru|      2026-09-03|
|          2006|        105|     Current|         Deposit|85000.0|   Mumbai|      2026-09-03|
|          2007|        106|     Savings|        Transfer|25000.0|    Delhi|      2026-09-04|
|          2009|        102|     Current|        Transfer|20000.0|Hyderabad|      2026-09-05|
|          2010|        108|     Savings|      Withdrawal| 3

In [0]:
Data is distributed based on account_type.
Improves parallelism.
Useful before groupBy, join, and aggregations on account_type.
Helps reduce data skew and optimize processing for large datasets.

18. Build a final banking pipeline producing clean, enriched transactions.



In [0]:
from pyspark.sql import functions as F

final_df = (
    df

    # Remove duplicate transaction IDs
    .dropDuplicates(["transaction_id"])

    # Handle null values
    .fillna({
        "amount": 0,
        "city": "Unknown"
    })

    # Convert date column to DateType
    .withColumn(
        "transaction_date",
        F.to_date("transaction_date", "yyyy-MM-dd")
    )

    # Extract Year, Month, Day
    .withColumn("year", F.year("transaction_date"))
    .withColumn("month", F.month("transaction_date"))
    .withColumn("day", F.dayofmonth("transaction_date"))

    # Create Risk Level
    .withColumn(
        "risk_level",
        F.when(F.col("amount") >= 100000, "HIGH")
         .when(F.col("amount") >= 50000, "MEDIUM")
         .otherwise("LOW")
    )

    # Add Transaction Fee (1%)
    .withColumn(
        "transaction_fee",
        F.round(F.col("amount") * 0.01, 2)
    )

    # Categorize Transactions
    .withColumn(
        "transaction_category",
        F.when(F.col("transaction_type") == "Deposit", "CREDIT")
         .otherwise("DEBIT")
    )

    # Repartition by account type
    .repartition("account_type")
)

final_df.show(truncate=False)

+--------------+-----------+------------+----------------+-------+---------+----------------+----+-----+---+----------+---------------+--------------------+
|transaction_id|customer_id|account_type|transaction_type|amount |city     |transaction_date|year|month|day|risk_level|transaction_fee|transaction_category|
+--------------+-----------+------------+----------------+-------+---------+----------------+----+-----+---+----------+---------------+--------------------+
|2004          |104        |Salary      |Deposit         |55000.0|Pune     |2026-09-02      |2026|9    |2  |MEDIUM    |550.0          |CREDIT              |
|2008          |107        |Salary      |Deposit         |60000.0|Bengaluru|2026-09-04      |2026|9    |4  |MEDIUM    |600.0          |CREDIT              |
|2014          |104        |Salary      |Transfer        |18000.0|Pune     |2026-09-07      |2026|9    |7  |LOW       |180.0          |DEBIT               |
|2002          |102        |Current     |Withdrawal      |

In [0]:
from pyspark.sql import functions as F

final_df = (
    df

    # Remove duplicate transactions
    .dropDuplicates(["transaction_id"])

    # Handle null values
    .fillna({
        "amount": 0,
        "city": "Unknown"
    })

    # Convert transaction date
    .withColumn(
        "transaction_date",
        F.to_date("transaction_date", "yyyy-MM-dd")
    )

    # Create risk level
    .withColumn(
        "risk_level",
        F.when(F.col("amount") >= 100000, "HIGH")
         .when(F.col("amount") >= 50000, "MEDIUM")
         .otherwise("LOW")
    )

    # Add transaction fee (1%)
    .withColumn(
        "transaction_fee",
        F.round(F.col("amount") * 0.01, 2)
    )

    # Extract year, month, day
    .withColumn("year", F.year("transaction_date"))
    .withColumn("month", F.month("transaction_date"))
    .withColumn("day", F.dayofmonth("transaction_date"))

    # Categorize transactions
    .withColumn(
        "transaction_category",
        F.when(F.col("transaction_type") == "Deposit", "CREDIT")
         .otherwise("DEBIT")
    )

    # Select required columns
    .select(
        "transaction_id",
        "customer_id",
        "account_type",
        "transaction_type",
        "amount",
        "transaction_fee",
        "risk_level",
        "transaction_category",
        "city",
        "transaction_date",
        "year",
        "month",
        "day"
    )

    # Sort by amount descending
    .orderBy(F.col("amount").desc())
)

final_df.show(truncate=False)

+--------------+-----------+------------+----------------+-------+---------------+----------+--------------------+---------+----------------+----+-----+---+
|transaction_id|customer_id|account_type|transaction_type|amount |transaction_fee|risk_level|transaction_category|city     |transaction_date|year|month|day|
+--------------+-----------+------------+----------------+-------+---------------+----------+--------------------+---------+----------------+----+-----+---+
|2006          |105        |Current     |Deposit         |85000.0|850.0          |MEDIUM    |CREDIT              |Mumbai   |2026-09-03      |2026|9    |3  |
|2011          |109        |Current     |Deposit         |75000.0|750.0          |MEDIUM    |CREDIT              |Pune     |2026-09-06      |2026|9    |6  |
|2008          |107        |Salary      |Deposit         |60000.0|600.0          |MEDIUM    |CREDIT              |Bengaluru|2026-09-04      |2026|9    |4  |
|2001          |101        |Savings     |Deposit         |

| # | Transformation | Type | Shuffle? |
|---|---|---|---|
| 1 | `select` | Narrow | No |
| 2, 3 | `filter` / `where` | Narrow | No |
| 4, 5 | `withColumn` (fee, net_amount) | Narrow | No |
| 6 | `withColumnRenamed` | Narrow | No |
| 7 | `drop` | Narrow | No |
| 8 | `distinct` | **Wide** | **Yes** – rows with the same value must be brought together to remove duplicates |
| 9 | `dropDuplicates` | **Wide** | **Yes** – same reason, on the key `transaction_id` |
| 10 | `orderBy` | **Wide** | **Yes** – range partitioning for a global sort |
| 11 | `groupBy` + `agg` | **Wide** | **Yes** – rows are shuffled by `account_type` (partial aggregation first, then Exchange) |
| 12 | `when/otherwise` | Narrow | No |
| 13 | `to_date`, `year`, `month`, `dayofmonth` | Narrow | No |
| 14 | `fillna` / `dropna` | Narrow | No |
| 15 | `unionByName` | Narrow | No – partitions are simply concatenated |
| 16 | `join` | **Wide** | **Yes** for a sort-merge/shuffle-hash join. If the small `customer_df` is broadcast (Spark's default for small tables), the join avoids shuffling the big side |
| 17 | `repartition` | **Wide** | **Yes** – full shuffle by hash of `account_type` |
| 17 | `coalesce` | Narrow | No – merges existing partitions |
| 18 | Final pipeline | Mixed | Shuffles at `dropDuplicates`, the `join` (unless broadcast) and the final `orderBy` |